# Studi Kasus: Otomatisasi Pemrosesan Data Aset & Inventarisasi Geospasial (*Geospatial Asset Management Automation*)

## 1. Latar Belakang & Skenario Masalah

Sebuah perusahaan pengelola aset (*Asset Management*) atau pemerintah daerah menangani ribuan data lokasi fisik (seperti titik tiang listrik, fasilitas publik, lahan, atau area tutupan vegetasi) yang diambil langsung dari survei lapangan (*field survey*).

Dalam operasional industri tradisional, proses pengolahan data spasial (seperti konversi format data, perhitungan luas/panjang, analisis *bounding box*, dan sinkronisasi ke basis data spasial) sering kali dilakukan secara manual melalui perangkat lunak desktop GIS. Hal ini memicu beberapa kendala:

1. **Ketidakefisienan Waktu**: Memindahkan dan mengedit data secara manual membutuhkan waktu berjam-jam.
2. **Risiko *Human Error***: Kesalahan input nilai atribut spasial atau ketidakcocokan format koordinat.
3. **Kebutuhan Integrasi *Cloud***: Kebutuhan akan *pipeline* data terotomatisasi (*Data Engineering for GIS*) yang siap diintegrasikan ke ekosistem *Cloud* / Web GIS (seperti PostGIS, QGIS, atau ArcGIS Online/Enterprise).

---

## 2. Solusi Berbasis Python Geospasial (`GeoPandas` & `Shapely`)

Mengembangkan skrip otomatisasi *Data Processing Pipeline* menggunakan pustaka Python geospasial (`GeoPandas`, `Shapely`) untuk:

* Membaca data spasial vektor (vektor titik/polygon lokasi aset).
* Melakukan pembersihan data (*data cleaning*) dan otomatisasi ekstraksi atribut spasial (seperti perhitungan luas area dalam meter persegi, penentuan koordinat centroid/titik tengah, dan identifikasi batas wilayah).
* Menyaring aset yang terindikasi memerlukan pemeliharaan berdasarkan kriteria tertentu.

---

## 3. Implementasi Kode Python

Berikut adalah kode Python yang mengimplementasikan otomatisasi analisis spasial data aset geospasial:

In [ ]:
import geopandas as gpd
from shapely.geometry import Point, Polygon
import pandas as pd

# 1. Simulasi Data Lapangan: Pemetaan Aset Taman / Fasilitas Publik (Sistem Koordinat UTM / Proyeksi Meter)
# EPSG:32748 (WGS 84 / UTM zone 48S - Wilayah Indonesia Bagian Barat / Bengkulu)
data_aset = {
    'asset_id': ['AST-101', 'AST-102', 'AST-103', 'AST-104'],
    'asset_name': ['Taman Kota A', 'Taman Bunga B', 'Fasilitas Olahraga C', 'Hutan Kota D'],
    'condition': ['Baik', 'Perlu Pemeliharaan', 'Baik', 'Perlu Pemeliharaan'],
    'geometry': [
        Polygon([(235000, 9578000), (235200, 9578000), (235200, 9578200), (235000, 9578200)]),  # 200m x 200m
        Polygon([(236000, 9579000), (236150, 9579000), (236150, 9579150), (236000, 9579150)]),  # 150m x 150m
        Polygon([(237000, 9577500), (237100, 9577500), (237100, 9577600), (237000, 9577600)]),  # 100m x 100m
        Polygon([(238000, 9580000), (238500, 9580000), (238500, 9580500), (238000, 9580500)])   # 500m x 500m
    ]
}

# 2. Mengonversi menjadi GeoDataFrame dengan Sistem Referensi Spasial (CRS) Projeksi Meter
gdf_assets = gpd.GeoDataFrame(data_aset, crs="EPSG:32748")

print("--- Data Aset Geospasial Awal ---")
print(gdf_assets[['asset_id', 'asset_name', 'condition']])

# 3. Otomatisasi Perhitungan Geometri Spasial
# a. Menghitung Luas Area Aset (Meter Persegi)
gdf_assets['area_m2'] = gdf_assets['geometry'].area

# b. Ekstraksi Koordinat Titik Tengah (Centroid) Aset untuk Keperluan Navigasi / Marker Map
gdf_assets['centroid_x'] = gdf_assets['geometry'].centroid.x
gdf_assets['centroid_y'] = gdf_assets['geometry'].centroid.y

# 4. Filter & Otomatisasi Rekomendasi Pemeliharaan
maintenance_queue = gdf_assets[gdf_assets['condition'] == 'Perlu Pemeliharaan'].copy()

print("\n--- Hasil Otomatisasi Analisis Spasial ---")
print(gdf_assets[['asset_id', 'asset_name', 'area_m2', 'centroid_x', 'centroid_y']])

print("\n--- Daftar Aset Terprioritas untuk Pemeliharaan ---")
for idx, row in maintenance_queue.iterrows():
    print(f"-> Aset ID: {row['asset_id']} | Nama: {row['asset_name']} | Luas Area: {row['area_m2']} m2 | Koordinat Centroid: ({row['centroid_x']:.2f}, {row['centroid_y']:.2f})")

# 5. Export Data Hasil Olahan ke GeoJSON (Format Standar Web GIS & Cloud Integration)
# gdf_assets.to_file("processed_assets.geojson", driver="GeoJSON")

---

## 4. Manfaat & Evaluasi Implementasi

* **Peningkatan Efisiensi**: Pemrosesan spasial yang biasanya memakan waktu manual dapat diselesaikan dalam hitungan detik (*automated GIS workflow*).
* **Standar Industri modern**: Skrip ini dapat dihubungkan ke API, basis data geospasial (*PostGIS*), atau platform *Cloud Engine* (seperti FME atau QGIS Python Plugin / PyQGIS) untuk membangun ekosistem tata kelola data aset yang terintegrasi secara otomatis (*seamless GIS integration*).